In [1]:
# Shaunak Dhande

In [ ]:
!pip install requests --upgrade --user
!pip install selenium beautifulsoup4 pandas openpyxl --user
!pip install webdriver_manager --user

In [57]:
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [17]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# URL to be loaded
url = "https://www.vcsheet.com/investors?stages=seed%7Cseries-a&sectors=fintech%7Cai-devtools%7Cgeneralist%7Cproptech&geographies=usa"
driver.get(url)

# Scroll to the end of the page to load all dynamic content
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # wait for new content to load
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height


page_source = driver.page_source
driver.quit()

soup = BeautifulSoup(page_source, "html.parser")

In [20]:
soup.text

'List of VC Investors (VC Sheet)\n\n\n\n\n\n\n\nBrowse Investors\n\n\nBrowse Funds\n\n\nBrowse Reporters\n\n\nBrowse Sheets\n\n\nHome\n\n\n2\n\n\n2\n\n\n2\n\n\nView your saves\n\n\nAll InvestorsBrowse all investors on VC Sheet.Browse InvestorsThank you! Your submission has been received!Oops! Something went wrong while submitting the form.Filters\n\n\nRounds They Invest InPre-SeedSeedSeries ASeries B+\n\n\nRounds They LeadSeedSeries ASeries B+Pre-Seed\n\n\nCheck Size Ranges$100K - $1M$0 - $100K$100K - $500K$500K - $1M$1M - $3M$3M - $10M$10M - $50M$50M+\n\n\nSector SpecialityAI / DevtoolsB2C/E-CommerceBiotechCPG/D2CClimateConsumerCybersecurityDeeptechDefense, Gov, & LegalDigital HealthE-CommerceEdtechEnterpriseFintechFood & Ag Future of WorkGaming/VRGeneralistHardwareLogisticsMarketplacesProptechUnder-represented foundersWeb3\n\n\nGeographic FocusUSAEuropeCanadaLatAmMENAAsia-PacificAfricaANZIsraelIndiaChinaSeedSeries ASeries B+Pre-Seed$100K - $1M$1M - $3M$3M - $10M$10M - $50M$100K - $50

In [50]:
investors_soup = soup.find_all(
    "div", class_="list-item vert-list less-spacing w-dyn-item"
)
base_url = "https://www.vcsheet.com"
all_investor_links = []
for i in investors_soup:
    investor_link = i.find("a", class_="full-click center w-inline-block")["href"]
    investor_complete_link = base_url + investor_link
    all_investor_links.append(investor_complete_link)

In [51]:
print(all_investor_links)

['https://www.vcsheet.com/who/ann-miura-ko', 'https://www.vcsheet.com/who/michael-gilroy', 'https://www.vcsheet.com/who/mike-volpi', 'https://www.vcsheet.com/who/ryan-freedman', 'https://www.vcsheet.com/who/satya-patel', 'https://www.vcsheet.com/who/zach-bratun-glennon', 'https://www.vcsheet.com/who/aaref-hilaly', 'https://www.vcsheet.com/who/abe-yokell', 'https://www.vcsheet.com/who/adriel-bercow', 'https://www.vcsheet.com/who/ajay-agarwal', 'https://www.vcsheet.com/who/amanda-robson', 'https://www.vcsheet.com/who/amit-karp', 'https://www.vcsheet.com/who/andy-chen', 'https://www.vcsheet.com/who/anish-acharya', 'https://www.vcsheet.com/who/aydin-senkut', 'https://www.vcsheet.com/who/ben-mathews', 'https://www.vcsheet.com/who/ben-sizes', 'https://www.vcsheet.com/who/charles-birnbaum', 'https://www.vcsheet.com/who/chetan-puttagunta', 'https://www.vcsheet.com/who/chris-cunningham', 'https://www.vcsheet.com/who/delian-asparouhov', 'https://www.vcsheet.com/who/ed-sim', 'https://www.vcsheet.

In [78]:
investors_data = []
for link in all_investor_links:
    response = requests.get(link)
    investor_soup = BeautifulSoup(response.content, "html.parser")
    name = (
        investor_soup.find("h1").get_text(strip=True)
        if investor_soup.find("h1")
        else ""
    )
    title = (
        investor_soup.find("div", class_="align-row wrapping").get_text(
            strip=True
        )
        if investor_soup.find("div", class_="align-row wrapping")
        else ""
    )
    email = (
        investor_soup.find(
            "a", class_="list-card contact-card email w-inline-block"
        )["href"]
        if investor_soup.find("a", "list-card contact-card email w-inline-block")
        else ""
    )
    email = email.replace("mailto:", "")
    linkedin = (
        investor_soup.find(
            "a", class_="list-card contact-card linkedin w-inline-block"
        )["href"]
        if investor_soup.find(
            "a", "list-card contact-card linkedin w-inline-block"
        )
        else ""
    )
    print(name)
    print(title)
    print(email)
    print(linkedin)
    print(link)
    investors_data.append(
        {
            "Name": name,
            "Title": title,
            "Email": email,
            "LinkedIn": linkedin,
            "Website": link,
        }
    )

Ann Miura-Ko
Co-Founding Partner@Floodgate
ann@floodgate.com
https://www.linkedin.com/in/amiura/
https://www.vcsheet.com/who/ann-miura-ko
Michael Gilroy
Co-COO of Growth, Co-Head of Fintech, General Partner@Coatue
mgilroy@coatue.com
https://www.linkedin.com/in/michaelbgilroy/
https://www.vcsheet.com/who/michael-gilroy
Mike Volpi
Partner@Index Ventures
m.volpi@indexventures.com
https://www.linkedin.com/in/mavolpi/
https://www.vcsheet.com/who/mike-volpi
Ryan Freedman
General Partner@Alpaca VC
ryan@alpaca.vc
https://www.linkedin.com/in/ryanfreedman1/
https://www.vcsheet.com/who/ryan-freedman
Satya Patel
Partner@Homebrew
satya@homebrew.co
https://www.linkedin.com/in/satyapatel/
https://www.vcsheet.com/who/satya-patel
Zach Bratun-Glennon
Founder and Partner@Gradient Ventures
zbg+external@gradient.com
https://www.linkedin.com/in/zachary-bratun-glennon-b4a16026/
https://www.vcsheet.com/who/zach-bratun-glennon
Aaref Hilaly
Partner@Bain Capital Ventures
ahilaly@baincapital.com
https://www.linke

Ali Jamal
Founding Partner@First Check Ventures
ajamal@firstcheckventures.com
https://www.linkedin.com/in/mralijamal/
https://www.vcsheet.com/who/ali-jamal
Andrew Endicott
Founding Partner@Gilgamesh Ventures
andrew@gilgameshvc.com
https://www.linkedin.com/in/andrewmendicott/
https://www.vcsheet.com/who/andrew-endicott
Andy McLoughlin
Managing Partner@Uncork Capital
andy.mcloughlin@softtechvc.com
https://www.linkedin.com/in/andymcloughlin/
https://www.vcsheet.com/who/andy-mcloughlin
Andy Weissman
Managing Partner@Union Square Ventures
andy@usv.com
https://www.linkedin.com/in/andrewweissman/
https://www.vcsheet.com/who/andy-weissman
Anna Patterson
Founder and Managing Partner@Gradient Ventures
anna+external@gradient.com
https://www.linkedin.com/in/anna-patterson-15921ba/
https://www.vcsheet.com/who/anna-patterson
Anne Lee Skates
Partner@Andreessen Horowitz
anne@a16z.com
https://www.linkedin.com/in/anneleeprinceton
https://www.vcsheet.com/who/anne-lee-skates
Annie Case
Partner@Kleiner Per

Immad Akhund
Investor@Immad Akhund Fund
immad@mercury.com
https://www.linkedin.com/in/iakhund/
https://www.vcsheet.com/who/immad-akhund
Itamar Novick
Founder@Recursive Ventures

https://www.linkedin.com/in/itamarnovick/
https://www.vcsheet.com/who/itamar-novick
Jack Abramowitz
Investment Associate@NextView Ventures
jack@nextviewventures.com
https://www.linkedin.com/in/jack-abramowitz-41a0a844/
https://www.vcsheet.com/who/jack-abramowitz
Jake Flomenberg
Partner@Wing Venture Capital
jake@wing.vc
http://www.linkedin.com/in/jacobflomenberg
https://www.vcsheet.com/who/jake-flomenberg
James Green
General Partner@CRV
james@crv.com
https://www.linkedin.com/in/james-green-201a4274/
https://www.vcsheet.com/who/james-green
Jan Hammer
General Partner@Index Ventures
j.hammer@indexventures.com
https://www.linkedin.com/in/hammerjan/
https://www.vcsheet.com/who/jan-hammer
Jason Warner
Managing Director@Redpoint
jwarner@redpoint.com
https://www.linkedin.com/in/jcw148/
https://www.vcsheet.com/who/jason-

Peter Levine
General Partner@Andreessen Horowitz
peter@a16z.com
https://www.linkedin.com/in/peter-levine-681386172/
https://www.vcsheet.com/who/peter-levine
Peter Xu
CEO & Managing Director, Plug & Play China@Plug N Play Ventures
p.xu@pnptc.com
https://www.linkedin.com/in/jieping-peter-xu-12aaa081/?originalSubdomain=cn
https://www.vcsheet.com/who/peter-xu
Ping Li
Partner@Accel
pli@accel.com
https://www.linkedin.com/in/pingli8/
https://www.vcsheet.com/who/ping-li
Rak Garg
Principal@Bain Capital Ventures
rgarg@baincapital.com
https://www.linkedin.com/in/rakgarg/
https://www.vcsheet.com/who/rak-garg
Raviraj Jain
Partner@Lightspeed Venture Partners
raviraj@lsvp.com
https://www.linkedin.com/in/ravirajjain/
https://www.vcsheet.com/who/raviraj-jain
Rebecca Kaden
General Partner@Union Square Ventures
rebecca@usv.com
https://www.linkedin.com/in/rebecca-kaden
https://www.vcsheet.com/who/rebecca-kaden
Rebecca Lynn
Co-Founder and General Partner@Canvas Ventures
rebecca@canvas.vc
https://www.linked

Adrian Mendoza
Founder and General Partner@Mendoza Ventures
adrian@mendoza-ventures.com
https://www.linkedin.com/in/adrianmendozavc/
https://www.vcsheet.com/who/adrian-mendoza
Adriana Saman
Director@Clocktower Technology Ventures
adriana@clocktowerventures.com
https://www.linkedin.com/in/adriana-saman-302ab95a/
https://www.vcsheet.com/who/adriana-saman
Aileen Lee
Founder and Managing Partner@Cowboy Ventures
aileen@cowboy.vc
https://www.linkedin.com/in/aileenwlee/
https://www.vcsheet.com/who/aileen-lee
Ajay Vashee
General Partner@IVP
ajay@ivp.com
https://www.linkedin.com/in/ajayvashee/
https://www.vcsheet.com/who/ajay-vashee
Akihiko Okamoto
Partner, Asia@Headline
aki@headline.com
https://www.linkedin.com/in/akioka/
https://www.vcsheet.com/who/akihiko-okamoto
Akshay Bhushan
Venture Partner@Lightspeed Venture Partners
akshay@lsip.com
https://www.linkedin.com/in/akshaybhushan/
https://www.vcsheet.com/who/akshay-bhushan
Al Sambar
General Partner@XRC Labs
al@xrclabs.com
linkedin.com/in/alsam

...

Michael Caso
Co-Founder, Managing Partner, & President@Rosecliff Ventures
caso@rosecliff.com
https://www.linkedin.com/in/michaelcaso
https://www.vcsheet.com/who/michael-caso
Michael DeSantis
Managing Director@Insight Partners
mdesantis@insightpartners.com
https://www.linkedin.com/in/michael-desantis-70749a2b/
https://www.vcsheet.com/who/michael-desantis
Michael Murphy
Co-Founder, Managing Partner & CEO@Rosecliff Ventures
murphy@rosecliff.com
https://www.linkedin.com/in/michael-murphy-40332817
https://www.vcsheet.com/who/michael-murphy
Michael Rogers
Partner, Venture Capital@Interplay

https://www.linkedin.com/in/mikejrogers/
https://www.vcsheet.com/who/michael-rogers
Michael Seibel
Managing Director, Early Stage and Group Partner@Y Combinator
michael@ycombinator.com
https://www.linkedin.com/in/mwseibel/
https://www.vcsheet.com/who/michael-seibel
Michael Silton
Managing Director@Act One Ventures
michael@actoneventures.com
https://www.linkedin.com/in/michaelsilton/
https://www.vcsheet.co

Pano Anthos
Founder and Managing Director@XRC Labs
pano@xrclabs.com
https://www.linkedin.com/in/panoanthos/
https://www.vcsheet.com/who/pano-anthos
Pat Matthews
Founder and CEO@Active Capital
pat@activecapital.com
https://www.linkedin.com/in/pamatthe/
https://www.vcsheet.com/who/pat-matthews
Patrick Backhouse
Partner@Greenoaks
patrick.backhouse@greenoakscap.com
https://www.linkedin.com/in/patrickbackhouse/
https://www.vcsheet.com/who/patrick-backhouse
Patrick Murphy
Co-Founder & General Partner@Tapestry VC

https://www.linkedin.com/in/pmurphyirl/
https://www.vcsheet.com/who/patrick-murphy
Patrick Riley
Co-Founder & Managing Partner@GAN Ventures
pat@gan.co
https://www.linkedin.com/in/rileypat
https://www.vcsheet.com/who/patrick-riley
Paul De Sadeleer
Investor@A* Capital
paul@a-star.co
https://www.linkedin.com/in/pauldesadeleer/
https://www.vcsheet.com/who/paul-de-sadeleer
Paul Ferri
Co-Founder@Matrix Partners
pferri@matrixpartners.com
https://www.linkedin.com/in/ferripaul
https://www.vc

Roger Ehrenberg
Founding Partner@IA Ventures
roger@iaventures.com
https://www.linkedin.com/in/rehrenberg/
https://www.vcsheet.com/who/roger-ehrenberg
Roland Fryer
Founding Partner@Equal Opportunity (EO) Ventures
rolandfryer@eoventures.com
https://www.linkedin.com/in/roland-fryer-9889b7183/
https://www.vcsheet.com/who/roland-fryer
Ruchi Sanghvi
Founder@South Park Commons
ruchi@southparkcommons.com
https://www.linkedin.com/in/rsanghvi/
https://www.vcsheet.com/who/ruchi-sanghvi
Ruchita Sinha
General Partner@AV8 Ventures
ruchita@av8.vc
https://www.linkedin.com/in/ruchita-sinha/
https://www.vcsheet.com/who/ruchita-sinha
Rudina Seseri
Founder & Managing Partner@Glasswing Ventures
rudina@glasswing.vc
https://www.linkedin.com/in/rudinaseseri/
https://www.vcsheet.com/who/rudina-seseri
Rusty Ralston
Co-Founder and General Partner@Swell Partners

https://www.linkedin.com/in/rustyr/
https://www.vcsheet.com/who/rusty-ralston
Ryan Broshar
Founder & Partner@Matchstick Ventures
ryan@matchstick.vc
http

Sriram Krishnan
Co-Founder and General Partner@Kearny Jackson
sriram@kearnyjackson.com
https://www.linkedin.com/in/sriramkrishnan/
https://www.vcsheet.com/who/sriram-krishnan-6
Stephanie Campbell
General Partner@The Artemis Fund
stephanie@theartemisfund.com
https://www.linkedin.com/in/stephanielcampbell/
https://www.vcsheet.com/who/stephanie-campbell
Stephanie Khoo
Partner@Nyca Partners
skhoo@nycapartners.com
https://www.linkedin.com/in/stephaniekkhoo/
https://www.vcsheet.com/who/stephanie-khoo
Stephen Bloch
General Partner@Canaan Partners
sbloch@canaan.com
https://www.linkedin.com/in/stephen-bloch-md-7a812616/
https://www.vcsheet.com/who/stephen-bloch
Stephen DiBartolomeo
Principal@Scout Ventures
stephen@scout.vc
https://www.linkedin.com/in/stephendibart/
https://www.vcsheet.com/who/stephen-dibartolomeo
Stephen McIntyre
Partner@Frontline Ventures
stephen@frontline.vc
https://www.linkedin.com/in/stephenmcintyre/
https://www.vcsheet.com/who/stephen-mcintyre
Stephen Oskoui
Managing Partn

Vic Singh
Co-Founder@Eniac Ventures
vic@eniac.vc
https://www.linkedin.com/in/vicsingh/
https://www.vcsheet.com/who/vic-singh
Victoria Treyger
General Partner@Felicis Ventures
victoria@felicis.com
https://www.linkedin.com/in/victoriatreyger/
https://www.vcsheet.com/who/victoria-treyger
Vijay Pande
General Partner@Andreessen Horowitz
vpande@a16z.com
https://www.linkedin.com/in/vijay-pande-phd-4b53342/
https://www.vcsheet.com/who/vijay-pande
Vikram Ramakrishnan
Principal@A* Capital
vikram@a-star.co
https://www.linkedin.com/in/vikramramakrishnan/
https://www.vcsheet.com/who/vikram-ramakrishnan
Villi Itchev
Managing Director@Two Sigma Ventures
villi@twosigmaventures.com
https://www.linkedin.com/in/villi04/
https://www.vcsheet.com/who/villi-itchev
Vineeta Agarwala
General Partner@Andreessen Horowitz
vineeta@a16z.com
https://www.linkedin.com/in/vineeta-agarwala-md-phd-674a591/
https://www.vcsheet.com/who/vineeta-agarwala
Vinny Pujji
Managing Partner@Left Lane Capital
vinny@leftlanecap.com
htt

In [79]:
# Convert the list to a DataFrame
df = pd.DataFrame(investors_data)

# Save to CSV and XLSX
df.to_excel("VCSheet_Query1.xlsx", index=False)

# Display the DataFrame
df.head()

,Name,Title,Email,LinkedIn,Website
0,Ann Miura-Ko,Co-Founding Partner@Floodgate,ann@floodgate.com,https://www.linkedin.com/in/amiura/,https://www.vcsheet.com/who/ann-miura-ko
1,Michael Gilroy,"Co-COO of Growth, Co-Head of Fintech, General ...",mgilroy@coatue.com,https://www.linkedin.com/in/michaelbgilroy/,https://www.vcsheet.com/who/michael-gilroy
2,Mike Volpi,Partner@Index Ventures,m.volpi@indexventures.com,https://www.linkedin.com/in/mavolpi/,https://www.vcsheet.com/who/mike-volpi
3,Ryan Freedman,General Partner@Alpaca VC,ryan@alpaca.vc,https://www.linkedin.com/in/ryanfreedman1/,https://www.vcsheet.com/who/ryan-freedman
4,Satya Patel,Partner@Homebrew,satya@homebrew.co,https://www.linkedin.com/in/satyapatel/,https://www.vcsheet.com/who/satya-patel
